# RQ3-06 — Merge 1,000 Human + 1,000 AI & Evaluate Frozen Detectors

Đây là notebook **external test cuối cùng**.

Input:
- `rq3_human_holdout_1000.csv` — 1,000 Human chưa từng train.
- `rq3_ai_1000.csv` — 1,000 AI mới từ 5 API.
- Các detector đã retrain và save từ notebook RQ3-02/03/04.

Output:
- `rq3_external_benchmark_2000.csv`
- Prediction từng model.
- Overall Accuracy / Precision / Recall / F1 / ROC-AUC / Human FPR.
- AI Recall theo generator, condition, prompt.
- Human FPR theo prompt.
- Performance drop so với internal test (`ΔF1`).

> Không tune threshold hay hyperparameter bằng benchmark này. Threshold mặc định class prediction của từng model được giữ nguyên.

## Checkpoint paths expected by this notebook

Notebook 13 **chỉ inference**, không train/fine-tune.

Expected outputs from the retrain notebooks:

```text
results/rq3_retrain_ml/models/
  tfidf_vectorizer.joblib
  feature_scaler.joblib
  logistic_regression.joblib
  random_forest.joblib
  lightgbm.joblib
  xgboost.joblib
  svm.joblib

results/rq3_retrain_modernbert_deberta/
  modernbert_final_model/
  deberta-v3_final_model/

results/rq3_retrain_distilbert_roberta/
  distilbert_best_final_model/
  roberta_best_final_model/
```

Nếu nhóm lưu checkpoint ở đường dẫn khác, chỉ sửa các path trong cell `ML_DIR` và `TRANSFORMERS`; **không load checkpoint RQ2 cũ**.

In [ ]:
# Nếu thiếu package:
# %pip install -U pandas numpy scipy scikit-learn joblib torch transformers accelerate

import json
import re
import hashlib
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

SEED = 42
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
if Path("/content/AI-Generated-Text-Essay-Detection").exists():
    PROJECT_ROOT = Path("/content/AI-Generated-Text-Essay-Detection")

RQ3_DIR = PROJECT_ROOT / "data" / "rq3"
RESULT_DIR = PROJECT_ROOT / "results" / "rq3_external"
RESULT_DIR.mkdir(parents=True, exist_ok=True)


def first_existing(candidates, label):
    p = next((x for x in candidates if x.exists()), None)
    if p is None:
        raise FileNotFoundError(f"Không tìm thấy {label}:\n" + "\n".join(map(str,candidates)))
    return p

HUMAN_FILE = first_existing([
    RQ3_DIR / "rq3_human_holdout_1000.csv",
    PROJECT_ROOT / "rq3_human_holdout_1000.csv",
    cwd / "rq3_human_holdout_1000.csv",
], "Human holdout")
AI_FILE = first_existing([
    RQ3_DIR / "rq3_ai_1000.csv",
    PROJECT_ROOT / "rq3_ai_1000.csv",
    cwd / "rq3_ai_1000.csv",
], "AI dataset")
TRAIN_FILE = first_existing([
    PROJECT_ROOT / "data" / "processed" / "data_with_features_minus_1000_human.csv",
    PROJECT_ROOT / "data_with_features_minus_1000_human.csv",
    cwd / "data_with_features_minus_1000_human.csv",
], "leakage-free train dataset")

print("Human:", HUMAN_FILE)
print("AI   :", AI_FILE)
print("Train:", TRAIN_FILE)

In [ ]:
human = pd.read_csv(HUMAN_FILE).copy()
ai = pd.read_csv(AI_FILE).copy()
assert len(human) == 1000, len(human)
assert len(ai) == 1000, len(ai)

# prompt_id lấy từ AI metadata, map ngược cho Human theo prompt_name.
prompt_map = ai[["prompt_id", "prompt_name"]].drop_duplicates()
if prompt_map["prompt_name"].duplicated().any():
    raise ValueError("Một prompt_name map tới nhiều prompt_id")
name_to_id = dict(zip(prompt_map["prompt_name"], prompt_map["prompt_id"]))

human_final = pd.DataFrame({
    "essay_id": human["essay_id"],
    "label": 0,
    "prompt_id": human["prompt_name"].map(name_to_id),
    "prompt_name": human["prompt_name"],
    "condition": "human",
    "generator_provider": "human",
    "generator_family": "human",
    "generator_model": "human",
    "final_provider": "human",
    "final_family": "human",
    "final_model": "human",
    "editor_provider": "",
    "editor_model": "",
    "parent_essay_id": "",
    "text": human["text_clean"].astype(str),
    "word_count": pd.to_numeric(human["word_count_clean"], errors="coerce"),
})

cols = list(human_final.columns)
ai_final = ai[cols].copy()
ai_final["label"] = 1

benchmark = pd.concat([human_final, ai_final], ignore_index=True)
benchmark = benchmark.sample(frac=1, random_state=SEED).reset_index(drop=True)

assert len(benchmark) == 2000
assert (benchmark["label"] == 0).sum() == 1000
assert (benchmark["label"] == 1).sum() == 1000
assert benchmark["essay_id"].nunique() == 2000
assert benchmark["prompt_id"].notna().all()

BENCHMARK_FILE = RQ3_DIR / "rq3_external_benchmark_2000.csv"
benchmark.to_csv(BENCHMARK_FILE, index=False)
print("Saved:", BENCHMARK_FILE)
print(benchmark["label"].value_counts())
print("\nAI generators:")
print(benchmark.loc[benchmark.label==1, "generator_family"].value_counts())
print("\nConditions:")
print(benchmark["condition"].value_counts())

In [ ]:
# Leakage audit theo normalized SHA-256 hash.
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

def text_hash(text):
    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()

train_texts = pd.read_csv(TRAIN_FILE, usecols=["text"])["text"].astype(str)
train_hashes = set(train_texts.map(text_hash))
bench_hashes = benchmark["text"].map(text_hash)

overlap_rows = benchmark[bench_hashes.isin(train_hashes)]
if len(overlap_rows):
    display(overlap_rows[["essay_id","label","prompt_name","generator_family"]].head(20))
    raise RuntimeError(f"Leakage audit FAIL: {len(overlap_rows)} benchmark rows overlap training")
print("✅ Leakage audit PASS: 0/2000 benchmark rows overlap retraining dataset")

## 1. Exact handcrafted features used by notebook 04/05

External AI text không có sẵn numeric features, nên tính lại **đúng công thức đã tạo `data_with_features.csv`**.

In [ ]:
FUNCTION_WORDS = set([
    'a','all','also','an','and','any','are','as','at','be','been',
    'can','do','down','even','every','for','from','had','has','have',
    'if','in','into','is','it','its','may','more','must','my','no',
    'of','on','one','only','or','our','shall','should','so','some',
    'that','the','their','then','there','things','this','to','up',
    'was','were','what','when','which','who','will','with','would','your'
])

FEATURE_COLS = [
    'avg_word_length', 'punctuation_ratio', 'burstiness',
    'word_count', 'avg_sentence_length', 'lexical_diversity',
    'sentence_count', 'function_word_ratio'
]

def extract_features(text):
    text = str(text)
    words = text.split()
    sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    sentence_lengths = [len(s.split()) for s in sentences]
    lower_words = text.lower().split()
    return {
        'avg_sentence_length': np.mean(sentence_lengths) if sentence_lengths else 0,
        'burstiness': np.std(sentence_lengths) if len(sentence_lengths) > 1 else 0,
        'lexical_diversity': len(set(words)) / len(words) if words else 0,
        'word_count': len(words),
        'sentence_count': len(sentences),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'punctuation_ratio': sum(1 for c in text if c in '.,!?') / len(text) if text else 0,
        'function_word_ratio': (sum(1 for w in lower_words if w in FUNCTION_WORDS) / len(lower_words)) if lower_words else 0,
    }

external_features = pd.DataFrame([extract_features(x) for x in benchmark["text"]])
assert set(FEATURE_COLS).issubset(external_features.columns)
display(external_features[FEATURE_COLS].head())

## 2. Evaluate Traditional ML

In [ ]:
ML_DIR = PROJECT_ROOT / "results" / "rq3_retrain_ml"
ML_MODEL_DIR = ML_DIR / "models"

vectorizer = joblib.load(ML_MODEL_DIR / "tfidf_vectorizer.joblib")
scaler = joblib.load(ML_MODEL_DIR / "feature_scaler.joblib")
X_text = vectorizer.transform(benchmark["text"].astype(str))
X_num = scaler.transform(external_features[FEATURE_COLS])
X_external_ml = hstack([X_text, csr_matrix(X_num)]).tocsr()
y_true = benchmark["label"].astype(int).to_numpy()

ML_MODELS = {
    "Logistic Regression": ("logistic_regression.joblib", "proba"),
    "Random Forest": ("random_forest.joblib", "proba"),
    "LightGBM": ("lightgbm.joblib", "proba"),
    "XGBoost": ("xgboost.joblib", "proba"),
    "SVM": ("svm.joblib", "decision"),
}

prediction_store = {}
for model_name, (filename, mode) in ML_MODELS.items():
    path = ML_MODEL_DIR / filename
    if not path.exists():
        print("Skip missing:", path)
        continue
    model = joblib.load(path)
    pred = model.predict(X_external_ml).astype(int)
    score = model.predict_proba(X_external_ml)[:,1] if mode == "proba" else model.decision_function(X_external_ml)
    prediction_store[model_name] = (pred, np.asarray(score, dtype=float))
    print("Loaded/evaluated:", model_name)

## 3. Evaluate saved Transformers

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

TRANSFORMERS = {
    "ModernBERT": {
        "path": PROJECT_ROOT / "results" / "rq3_retrain_modernbert_deberta" / "modernbert_final_model",
        "max_length": 256,
        "return_token_type_ids": None,
    },
    "DeBERTa-v3": {
        "path": PROJECT_ROOT / "results" / "rq3_retrain_modernbert_deberta" / "deberta-v3_final_model",
        "max_length": 256,
        "return_token_type_ids": False,
    },
    "DistilBERT": {
        "path": PROJECT_ROOT / "results" / "rq3_retrain_distilbert_roberta" / "distilbert_best_final_model",
        "max_length": 256,
        "return_token_type_ids": None,
    },
    "RoBERTa": {
        "path": PROJECT_ROOT / "results" / "rq3_retrain_distilbert_roberta" / "roberta_best_final_model",
        "max_length": 256,
        "return_token_type_ids": None,
    },
}

class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length, return_token_type_ids=None):
        self.texts = list(map(str, texts))
        self.tok = tokenizer
        self.max_length = max_length
        self.return_token_type_ids = return_token_type_ids
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i], truncation=True, padding="max_length",
            max_length=self.max_length,
            **({} if self.return_token_type_ids is None else {"return_token_type_ids": self.return_token_type_ids}),
            return_tensors="pt",
        )
        return {k:v.squeeze(0) for k,v in enc.items()}

@torch.no_grad()
def transformer_predict(model_path, max_length, return_token_type_ids=None, batch_size=16):
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(DEVICE)
    model.eval()
    ds = InferenceDataset(benchmark["text"], tok, max_length, return_token_type_ids)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    probs=[]
    for batch in loader:
        batch = {k:v.to(DEVICE) for k,v in batch.items()}
        logits = model(**batch).logits
        probs.append(torch.softmax(logits, dim=1)[:,1].cpu().numpy())
    prob = np.concatenate(probs)
    pred = (prob >= 0.5).astype(int)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return pred, prob

for model_name, cfg in TRANSFORMERS.items():
    if not cfg["path"].exists():
        print("Skip missing transformer:", cfg["path"])
        continue
    pred, score = transformer_predict(
        cfg["path"], cfg["max_length"], cfg["return_token_type_ids"]
    )
    prediction_store[model_name] = (pred, score)
    print("Loaded/evaluated:", model_name)

## 4. Overall RQ3 metrics + detailed stress-test tables

In [ ]:
def overall_metrics(model_name, pred, score):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, score),
        "Human_FPR": fp / (fp + tn),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }

summary=[]
by_generator=[]
by_condition=[]
human_by_prompt=[]

for model_name, (pred, score) in prediction_store.items():
    summary.append(overall_metrics(model_name, pred, score))
    tmp = benchmark.copy()
    tmp["prediction"] = pred
    tmp["score_ai"] = score

    ai_tmp = tmp[tmp.label == 1]
    for g, part in ai_tmp.groupby("generator_family"):
        by_generator.append({
            "Model": model_name, "Generator": g, "N": len(part),
            "AI_Recall": part["prediction"].mean(),
            "Mean_score_ai": part["score_ai"].mean(),
        })
    for cond, part in ai_tmp.groupby("condition"):
        by_condition.append({
            "Model": model_name, "Condition": cond, "N": len(part),
            "AI_Recall": part["prediction"].mean(),
            "Mean_score_ai": part["score_ai"].mean(),
        })
    human_tmp = tmp[tmp.label == 0]
    for prompt, part in human_tmp.groupby("prompt_name"):
        human_by_prompt.append({
            "Model": model_name, "Prompt": prompt, "N": len(part),
            "Human_FPR": part["prediction"].mean(),
            "Mean_score_ai": part["score_ai"].mean(),
        })

    pd.DataFrame({
        "essay_id": benchmark["essay_id"],
        "label": y_true,
        "prediction": pred,
        "score_ai": score,
        "prompt_name": benchmark["prompt_name"],
        "condition": benchmark["condition"],
        "generator_family": benchmark["generator_family"],
    }).to_csv(RESULT_DIR / f"{model_name.lower().replace(' ','_').replace('-','_')}_rq3_predictions.csv", index=False)

summary_df = pd.DataFrame(summary).sort_values("F1", ascending=False)
by_generator_df = pd.DataFrame(by_generator)
by_condition_df = pd.DataFrame(by_condition)
human_prompt_df = pd.DataFrame(human_by_prompt)

display(summary_df)

## 5. Performance drop: internal test → RQ3 external

In [ ]:
internal_frames=[]
for p in [
    PROJECT_ROOT / "results" / "rq3_retrain_ml" / "ml_internal_results.csv",
    PROJECT_ROOT / "results" / "rq3_retrain_modernbert_deberta" / "modernbert_deberta_internal_results.csv",
    PROJECT_ROOT / "results" / "rq3_retrain_distilbert_roberta" / "distilbert_roberta_internal_results.csv",
]:
    if p.exists():
        x = pd.read_csv(p)
        if "Model" in x.columns and "F1" in x.columns:
            internal_frames.append(x[["Model","F1"]].rename(columns={"F1":"Internal_F1"}))

if internal_frames:
    internal = pd.concat(internal_frames, ignore_index=True).drop_duplicates("Model", keep="last")
    summary_df = summary_df.merge(internal, on="Model", how="left")
    summary_df["Delta_F1_external_minus_internal"] = summary_df["F1"] - summary_df["Internal_F1"]
else:
    summary_df["Internal_F1"] = np.nan
    summary_df["Delta_F1_external_minus_internal"] = np.nan

SUMMARY_FILE = RESULT_DIR / "rq3_external_summary.csv"
summary_df.to_csv(SUMMARY_FILE, index=False)
by_generator_df.to_csv(RESULT_DIR / "rq3_by_generator.csv", index=False)
by_condition_df.to_csv(RESULT_DIR / "rq3_by_condition.csv", index=False)
human_prompt_df.to_csv(RESULT_DIR / "rq3_human_fpr_by_prompt.csv", index=False)

print("Saved:", SUMMARY_FILE)
display(summary_df.sort_values("F1", ascending=False))

## Reporting checklist

Báo cáo RQ3 nên có:

1. **Overall external**: F1, AI Recall, Human FPR, AUROC.
2. **Seen-family/version shift**: GPT, Claude, Gemini/Google, Llama.
3. **Completely unseen family**: DeepSeek.
4. **Prompt/style shift**: Standard vs Natural vs Student-style vs Post-edited.
5. `ΔF1 = F1_external - F1_internal` (âm nghĩa là performance drop).
6. Không chọn model/threshold dựa trên RQ3 rồi báo lại trên cùng RQ3 như test độc lập.